# FactorForge v1 — Kaggle Training Notebook

Run 2: SGN 38K+ CDS + GC penalty. Run cells in order with GPU T4 x2 enabled.

In [ ]:
# Cell 1: Setup working directory
WORK_DIR = "/kaggle/working/factorforge_data"
import os
os.makedirs(WORK_DIR, exist_ok=True)
print(f"Work dir: {WORK_DIR}")


In [ ]:
# Cell 2: Clone repo, install dependencies, install CD-HIT
import os, shutil

if os.path.exists('/kaggle/working/factorforge'):
    shutil.rmtree('/kaggle/working/factorforge')

!git clone https://github.com/eijex/factorforge.git /kaggle/working/factorforge -q
!pip install -e /kaggle/working/factorforge -q
!pip install fair-esm -q
!apt-get install -y cd-hit -q
print("Done")


In [ ]:
# Cell 3: Load Run 2 config and map paths to working directory
import yaml

CONFIG_PATH = "/kaggle/working/factorforge/configs/v3_training_config_run2.yml"
with open(CONFIG_PATH) as f:
    run2_cfg = yaml.safe_load(f)

run2_cfg["paths"]["embeddings_dir"] = f"{WORK_DIR}/data/embeddings/per_token_run2"
run2_cfg["paths"]["checkpoint_dir"] = f"{WORK_DIR}/checkpoints_run2"
run2_cfg["paths"]["model_output"]   = f"{WORK_DIR}/model_run2"
run2_cfg["data"]["train_file"] = f"{WORK_DIR}/data/training/train_v3_run2.jsonl"
run2_cfg["data"]["eval_file"]  = f"{WORK_DIR}/data/training/eval_v3_run2.jsonl"
print("Config ready:", run2_cfg["bart"])


In [ ]:
# Cell 4: Fetch SGN CDS, filter by CAI, homology split (CD-HIT 0.7)
import os
os.makedirs(f"{WORK_DIR}/data/raw", exist_ok=True)
os.makedirs(f"{WORK_DIR}/data/training", exist_ok=True)

!cd /kaggle/working/factorforge && python scripts/1_data_preparation/fetch_sgn_cds.py     --output "{WORK_DIR}/data/raw/sgn_nbenthamiana_cds.fasta"

!cd /kaggle/working/factorforge && python scripts/1_data_preparation/filter_by_cai.py     --input "{WORK_DIR}/data/raw/sgn_nbenthamiana_cds.fasta"     --output "{WORK_DIR}/data/training/training_pairs_v3_run2.jsonl"     --threshold 0.55

!cd /kaggle/working/factorforge && python scripts/1_data_preparation/homology_split.py     --input "{WORK_DIR}/data/training/training_pairs_v3_run2.jsonl"     --train-output "{WORK_DIR}/data/training/train_v3_run2.jsonl"     --eval-output "{WORK_DIR}/data/training/eval_v3_run2.jsonl"     --verify


In [ ]:
# Cell 5: Extract per-token ESM2 embeddings (~1-2 hours on T4)
import sys, json
from pathlib import Path
sys.path.insert(0, "/kaggle/working/factorforge/scripts/1_data_preparation")
from extract_per_token_esm2 import extract_embeddings

run2_fasta = Path(f"{WORK_DIR}/data/raw/sgn_run2_aa.fasta")
with open(run2_cfg["data"]["train_file"]) as th,      open(run2_cfg["data"]["eval_file"]) as eh,      run2_fasta.open("w") as fh:
    for line in list(th) + list(eh):
        row = json.loads(line)
        fh.write(f">{row['id']}
{row['aa_seq']}
")

extract_embeddings(
    fasta_path=str(run2_fasta),
    output_dir=run2_cfg["paths"]["embeddings_dir"],
    model_name=run2_cfg["esm2"]["model_name"],
    batch_size=run2_cfg["esm2"]["batch_size"],
)


In [ ]:
# Cell 6: Train with GC penalty (~2-4 hours on T4)
import importlib.util, sys

training_dir = "/kaggle/working/factorforge/scripts/3_training"

spec = importlib.util.spec_from_file_location("dataset", f"{training_dir}/dataset.py")
dataset_mod = importlib.util.module_from_spec(spec)
sys.modules["dataset"] = dataset_mod
spec.loader.exec_module(dataset_mod)

spec2 = importlib.util.spec_from_file_location("train_v3_esm2_bart", f"{training_dir}/train_v3_esm2_bart.py")
train_mod = importlib.util.module_from_spec(spec2)
spec2.loader.exec_module(train_mod)

train_mod.train(run2_cfg, dry_run=False)


In [ ]:
# Cell 7: Evaluate — GFP / CD47 / mCherry CAI and GC%
import sys, torch
sys.path.insert(0, "/kaggle/working/factorforge/src")
from factorforge.engines.v3.metrics import compute_cai, compute_gc, load_codon_usage_table
from factorforge.engines.v3.pipeline import V3Pipeline

codon_table = load_codon_usage_table()
pipeline = V3Pipeline(checkpoint_dir=run2_cfg["paths"]["model_output"])

TEST_PROTEINS = {
    "GFP":     "MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVPWPTLVTTFSYGVQCFSRYPDHMKQHDFFKSAMPEGYVQERTIFFKDDGNYKTRAEVKFEGDTLVNRIELKGIDFKEDGNILGHKLEYNYNSHVYIMADKQKNGIKVNFKIRHNIEDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQSALSKDPNEKRDHMVLLEFVTAAGITHGMDELYK",
    "CD47":    "MWPLVAALLLGSACCGSAQLLFNKTKSVEHSDGDLVNEVDGSNFTVSLEPGGRRITMQLKPKDGEFIQSPTRTLDQFTFVQLNESKEVEGMAYRMV",
    "mCherry": "MVSKGEEDNMAIIKEFMRFKVHMEGSVNGHEFEIEGEGEGRPYEGTQTAKLKVTKGGPLPFAWDILSPQFMYGSKAYVKHPADIPDYLKLSFPEGFKWERVMNFEDGGVVTVTQDSSLQDGEFIYKVKLRGTNFPSDGPVMQKKTMGWEASSERMYPEDGALKGEIKQRLKLKDGGHYDAEVKTTYKAKKPVQLPGAYNVNIKLDITSHNEDYTIVEQYERAEGRHSTGGMDELYK",
}

print(f"{'Protein':<10} {'CAI':>6} {'GC%':>6}")
print("-" * 26)
for name, seq in TEST_PROTEINS.items():
    result = pipeline.run(seq)
    cai = compute_cai(result.sequence, codon_table)
    gc  = compute_gc(result.sequence)
    print(f"{name:<10} {cai:>6.4f} {gc:>6.1f}%")


## Run 4: v2 pseudo-label full training prep

This section prepares v2 pseudo-label Run 4 data, reuses or creates per-token ESM2 embeddings, trains with `configs/v3_training_config_run4.yml`, logs loss components, and writes constrained-decoding outputs to `experiments/results/run4_full/`.

In [ ]:
# Run 4 Cell 1: Configure paths
import os, json, yaml
from pathlib import Path

REPO_DIR = Path('/kaggle/working/factorforge')
WORK_DIR = Path(globals().get('WORK_DIR', '/kaggle/working/factorforge_data'))
RUN4_DIR = WORK_DIR / 'run4'
RUN4_DATA_DIR = WORK_DIR / 'data' / 'training'
RUN4_RAW_CDS = WORK_DIR / 'data' / 'raw' / 'sgn_nbenthamiana_cds.fasta'
RUN4_EMBEDDINGS = WORK_DIR / 'data' / 'embeddings' / 'per_token_run2'
RUN4_RESULTS = REPO_DIR / 'experiments' / 'results' / 'run4_full'

# If a Drive-like mount exists, use it; otherwise keep export artifacts in Kaggle working output.
DRIVE_ROOT = Path('/kaggle/working/drive/MyDrive')
RUN4_DRIVE_DIR = (DRIVE_ROOT / 'factorforge' / 'run4') if DRIVE_ROOT.exists() else Path('/kaggle/working/factorforge_run4_drive_export')
for directory in [RUN4_DIR, RUN4_DATA_DIR, RUN4_EMBEDDINGS, RUN4_RESULTS, RUN4_DRIVE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

RUN4_CONFIG = RUN4_DIR / 'v3_training_config_run4.kaggle.yml'
print('Run 4 data:', RUN4_DATA_DIR)
print('Run 4 embeddings:', RUN4_EMBEDDINGS)
print('Run 4 results:', RUN4_RESULTS)
print('Run 4 checkpoint/log export:', RUN4_DRIVE_DIR)


In [ ]:
# Run 4 Cell 2: Generate v2 pseudo-label train/eval JSONL from SGN CDS FASTA
import subprocess, json, shutil
from pathlib import Path

fetch_script = REPO_DIR / 'scripts' / '1_data_preparation' / 'fetch_sgn_cds.py'
pseudolabel_script = REPO_DIR / 'scripts' / '1_data_preparation' / 'generate_v2_pseudolabels.py'
if not pseudolabel_script.exists() or not fetch_script.exists():
    print('FactorForge checkout is missing Run 4 scripts; refreshing /kaggle/working/factorforge from origin/main ...')
    if (REPO_DIR / '.git').exists():
        subprocess.run(['git', 'fetch', 'origin', 'main'], cwd=REPO_DIR, check=True)
        subprocess.run(['git', 'reset', '--hard', 'origin/main'], cwd=REPO_DIR, check=True)
    else:
        if REPO_DIR.exists():
            shutil.rmtree(REPO_DIR)
        subprocess.run(['git', 'clone', 'https://github.com/eijex/factorforge.git', str(REPO_DIR)], check=True)
    subprocess.run(['python', '-m', 'pip', 'install', '-e', str(REPO_DIR), '-q'], check=True)

fetch_script = REPO_DIR / 'scripts' / '1_data_preparation' / 'fetch_sgn_cds.py'
pseudolabel_script = REPO_DIR / 'scripts' / '1_data_preparation' / 'generate_v2_pseudolabels.py'
if not pseudolabel_script.exists():
    raise FileNotFoundError(f'Missing pseudo-label script after repo refresh: {pseudolabel_script}')

# Prefer the Run 2/Run 4 working FASTA, but make this cell self-contained on Kaggle.
candidate_fastas = [
    RUN4_RAW_CDS,
    WORK_DIR / 'data' / 'raw' / 'sgn_nbenthamiana_cds.fasta',
    Path('/kaggle/working/factorforge_data/data/raw/sgn_nbenthamiana_cds.fasta'),
]
for input_root in Path('/kaggle/input').glob('*') if Path('/kaggle/input').exists() else []:
    candidate_fastas.extend(input_root.rglob('sgn_nbenthamiana_cds.fasta'))
    candidate_fastas.extend(input_root.rglob('*sgn*cds*.fasta'))
    candidate_fastas.extend(input_root.rglob('*sgn*cds*.fa'))

source_fasta = next((path for path in candidate_fastas if path.exists()), None)
if source_fasta is None:
    if not fetch_script.exists():
        raise FileNotFoundError(f'Missing SGN CDS FASTA and fetch script: {fetch_script}')
    print('SGN CDS FASTA not found locally; fetching with fetch_sgn_cds.py ...')
    RUN4_RAW_CDS.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        'python', str(fetch_script),
        '--output', str(RUN4_RAW_CDS),
    ], cwd=REPO_DIR, check=True)
    source_fasta = RUN4_RAW_CDS
elif source_fasta != RUN4_RAW_CDS:
    RUN4_RAW_CDS.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source_fasta, RUN4_RAW_CDS)
    source_fasta = RUN4_RAW_CDS

print(f'Using SGN CDS FASTA: {source_fasta}')

cmd = [
    'python', str(pseudolabel_script),
    '--fasta', str(source_fasta),
    '--input-kind', 'cds',
    '--output', str(RUN4_DATA_DIR / 'run4_pseudolabels.jsonl'),
    '--train-output', str(RUN4_DATA_DIR / 'run4_pseudolabels_train.jsonl'),
    '--eval-output', str(RUN4_DATA_DIR / 'run4_pseudolabels_eval.jsonl'),
    '--train-split', '0.9',
    '--seed', '42',
    '--profile', 'high_cai',
    '--max-length', '512',
    '--limit', '38754',
]
result = subprocess.run(cmd, cwd=REPO_DIR, check=True, text=True, capture_output=True)
print(result.stdout)
stdout = result.stdout.strip()
json_start = stdout.find('{')
json_end = stdout.rfind('}')
if json_start == -1 or json_end == -1 or json_end < json_start:
    raise ValueError(f'Pseudo-label generation did not emit a JSON summary. stdout={stdout!r}')
run4_generation_summary = json.loads(stdout[json_start:json_end + 1])
print({'run4_generation_summary': run4_generation_summary})


In [ ]:
# Run 4 Cell 3: Verify pseudo-label schema and validator pass rate
import json
from pathlib import Path

# Recover Run 4 path variables if the kernel was restarted or Cell 1 was skipped.
if 'REPO_DIR' not in globals():
    REPO_DIR = Path('/kaggle/working/factorforge')
if 'WORK_DIR' not in globals():
    WORK_DIR = Path('/kaggle/working/factorforge_data')
if 'RUN4_DIR' not in globals():
    RUN4_DIR = WORK_DIR / 'run4'
if 'RUN4_DATA_DIR' not in globals():
    RUN4_DATA_DIR = WORK_DIR / 'data' / 'training'
if 'RUN4_EMBEDDINGS' not in globals():
    RUN4_EMBEDDINGS = WORK_DIR / 'data' / 'embeddings' / 'per_token_run2'
if 'RUN4_RESULTS' not in globals():
    RUN4_RESULTS = REPO_DIR / 'experiments' / 'results' / 'run4_full'
if 'RUN4_DRIVE_DIR' not in globals():
    DRIVE_ROOT = Path('/kaggle/working/drive/MyDrive')
    RUN4_DRIVE_DIR = (DRIVE_ROOT / 'factorforge' / 'run4') if DRIVE_ROOT.exists() else Path('/kaggle/working/factorforge_run4_drive_export')
for directory in [RUN4_DIR, RUN4_DATA_DIR, RUN4_EMBEDDINGS, RUN4_RESULTS, RUN4_DRIVE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

train_file = RUN4_DATA_DIR / 'run4_pseudolabels_train.jsonl'
eval_file = RUN4_DATA_DIR / 'run4_pseudolabels_eval.jsonl'
if not train_file.exists() or not eval_file.exists():
    raise FileNotFoundError(
        f'Missing Run 4 pseudo-label split files under {RUN4_DATA_DIR}. '
        'Run Run 4 Cell 2 first, then rerun this cell.'
    )

required = {'protein_id', 'amino_acid_sequence', 'protein_sequence', 'dna_sequence', 'dna_seq', 'codon_sequence', 'validator'}

def inspect_jsonl(path):
    total = 0
    passed = 0
    with path.open() as handle:
        for line in handle:
            row = json.loads(line)
            missing = required - set(row)
            if missing:
                raise ValueError(f'{path} row is missing {missing}')
            total += 1
            passed += int(bool(row['validator']['passed']))
    return total, passed

train_total, train_passed = inspect_jsonl(train_file)
eval_total, eval_passed = inspect_jsonl(eval_file)
print({'train_total': train_total, 'train_passed': train_passed, 'eval_total': eval_total, 'eval_passed': eval_passed})
assert train_total >= 30000, 'Run 4 target expects at least 30,000 train pseudo-labels'
assert train_passed == train_total and eval_passed == eval_total


In [ ]:
# Run 4 Cell 4: Reuse or generate per-token ESM2 embeddings
import json, subprocess
from pathlib import Path

# Recover Run 4 path variables if this cell is run after a kernel restart.
if 'REPO_DIR' not in globals():
    REPO_DIR = Path('/kaggle/working/factorforge')
if 'WORK_DIR' not in globals():
    WORK_DIR = Path('/kaggle/working/factorforge_data')
if 'RUN4_DATA_DIR' not in globals():
    RUN4_DATA_DIR = WORK_DIR / 'data' / 'training'
if 'RUN4_EMBEDDINGS' not in globals():
    RUN4_EMBEDDINGS = WORK_DIR / 'data' / 'embeddings' / 'per_token_run2'
RUN4_DATA_DIR.mkdir(parents=True, exist_ok=True)
RUN4_EMBEDDINGS.mkdir(parents=True, exist_ok=True)

if 'train_file' not in globals():
    train_file = RUN4_DATA_DIR / 'run4_pseudolabels_train.jsonl'
if 'eval_file' not in globals():
    eval_file = RUN4_DATA_DIR / 'run4_pseudolabels_eval.jsonl'
if not train_file.exists() or not eval_file.exists():
    raise FileNotFoundError(
        f'Missing Run 4 pseudo-label split files under {RUN4_DATA_DIR}. '
        'Run Run 4 Cell 2 and Cell 3 first.'
    )

run4_fasta = RUN4_DATA_DIR / 'run4_pseudolabels_aa.fasta'
train_total = 0
eval_total = 0
with train_file.open() as train_handle, eval_file.open() as eval_handle, run4_fasta.open('w') as fasta_handle:
    for line in train_handle:
        row = json.loads(line)
        fasta_handle.write('>{}\n{}\n'.format(row['protein_id'], row['amino_acid_sequence']))
        train_total += 1
    for line in eval_handle:
        row = json.loads(line)
        fasta_handle.write('>{}\n{}\n'.format(row['protein_id'], row['amino_acid_sequence']))
        eval_total += 1

expected_count = train_total + eval_total
existing_count = len(list(RUN4_EMBEDDINGS.glob('*.pt')))
print({'expected_embeddings': expected_count, 'existing_embeddings': existing_count, 'fasta': str(run4_fasta)})
if existing_count < expected_count:
    print('Generating missing ESM2 embeddings on Kaggle GPU/CPU. This can take 1-2 hours on T4.')
    try:
        import esm  # noqa: F401
    except ModuleNotFoundError:
        print('Installing fair-esm for ESM2 embedding extraction ...')
        subprocess.run(['python', '-m', 'pip', 'install', 'fair-esm', '-q'], check=True)
    subprocess.run([
        'python', str(REPO_DIR / 'scripts' / '1_data_preparation' / 'extract_per_token_esm2.py'),
        '--fasta', str(run4_fasta),
        '--output', str(RUN4_EMBEDDINGS),
        '--model', 'esm2_t6_8M_UR50D',
        '--batch-size', '16',
        '--max-length', '512',
    ], cwd=REPO_DIR, check=True)
else:
    print('Reusing existing Run 2 per-token embeddings.')


In [ ]:
# Run 4 Cell 5: Materialize Kaggle Run 4 config
base_cfg_path = REPO_DIR / 'configs' / 'v3_training_config_run4.yml'
with base_cfg_path.open() as handle:
    run4_cfg = yaml.safe_load(handle)

run4_cfg['paths']['embeddings_dir'] = str(RUN4_EMBEDDINGS)
run4_cfg['paths']['checkpoint_dir'] = str(RUN4_DRIVE_DIR / 'checkpoints')
run4_cfg['paths']['model_output'] = str(RUN4_DRIVE_DIR / 'model')
run4_cfg['paths']['training_data'] = str(train_file)
run4_cfg['data']['train_file'] = str(train_file)
run4_cfg['data']['eval_file'] = str(eval_file)

with RUN4_CONFIG.open('w') as handle:
    yaml.safe_dump(run4_cfg, handle, sort_keys=False)

print(yaml.safe_dump({
    'gc_low': run4_cfg['loss']['gc_low'],
    'gc_high': run4_cfg['loss']['gc_high'],
    'expected_log_cai_weight': run4_cfg['loss']['expected_log_cai_weight'],
    'require_synonym_mask': run4_cfg['data']['require_synonym_mask'],
    'max_steps': run4_cfg['training']['max_steps'],
    'config': str(RUN4_CONFIG),
}, sort_keys=False))


In [ ]:
# Run 4 Cell 6: Train with loss component logging
import subprocess

loss_log = RUN4_DRIVE_DIR / 'run4_loss_log.csv'
subprocess.run([
    'python', str(REPO_DIR / 'scripts' / '3_training' / 'train_v3_esm2_bart.py'),
    '--config', str(RUN4_CONFIG),
    '--loss-log', str(loss_log),
], cwd=REPO_DIR, check=True)

print('Run 4 loss log:', loss_log)


In [ ]:
# Run 4 Cell 7: Constrained decode eval set and save Run 4 full outputs
import shutil, subprocess

# Full eval decodes 3,876 sequences and can take hours. Set to 100 or 200 for a quick sanity check,
# then set back to None for the final full Run 4 output.
RUN4_EVAL_LIMIT = None

checkpoint = RUN4_DRIVE_DIR / 'model' / 'pytorch_model.pt'
if not checkpoint.exists():
    raise FileNotFoundError(f'Missing trained model checkpoint: {checkpoint}')

eval_cmd = [
    'python', str(REPO_DIR / 'scripts' / '4_evaluation' / 'run4_constrained_eval.py'),
    '--config', str(RUN4_CONFIG),
    '--checkpoint', str(checkpoint),
    '--eval-file', str(eval_file),
    '--embeddings-dir', str(RUN4_EMBEDDINGS),
    '--output-dir', str(RUN4_RESULTS),
    '--progress-every', '25',
]
if RUN4_EVAL_LIMIT is not None:
    eval_cmd.extend(['--limit', str(RUN4_EVAL_LIMIT)])

subprocess.run(eval_cmd, cwd=REPO_DIR, check=True)

shutil.copy2(RUN4_DRIVE_DIR / 'run4_loss_log.csv', RUN4_RESULTS / 'run4_loss_log.csv')
for name in ['run4_loss_log.csv', 'run4_comparison.csv', 'run4_summary.json']:
    path = RUN4_RESULTS / name
    print(path, path.exists(), path.stat().st_size if path.exists() else None)


## Run 4B Smoke: mixed teacher recovery

Run 4B tests mixed pseudo-label teachers and stronger GC penalties. This is smoke-only; do not use these cells for full training.

In [ ]:
# Run 4B Cell 1: Setup, refresh repo, and generate mixed pseudo-labels
import json, shutil, subprocess, yaml
from pathlib import Path

REPO_DIR = Path('/kaggle/working/factorforge')
WORK_DIR = Path(globals().get('WORK_DIR', '/kaggle/working/factorforge_data'))
RUN4B_DIR = WORK_DIR / 'run4b'
RUN4B_DATA_DIR = WORK_DIR / 'data' / 'training'
RUN4B_EMBEDDINGS = WORK_DIR / 'data' / 'embeddings' / 'per_token_run2'
RUN4B_RESULTS = REPO_DIR / 'experiments' / 'results' / 'run4b_smoke'
RUN4B_EXPORT = Path('/kaggle/working/factorforge_run4b_smoke_export')
RUN4_RAW_CDS = WORK_DIR / 'data' / 'raw' / 'sgn_nbenthamiana_cds.fasta'
for directory in [RUN4B_DIR, RUN4B_DATA_DIR, RUN4B_EMBEDDINGS, RUN4B_RESULTS, RUN4B_EXPORT, RUN4_RAW_CDS.parent]:
    directory.mkdir(parents=True, exist_ok=True)

if not (REPO_DIR / '.git').exists():
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(['git', 'clone', 'https://github.com/eijex/factorforge.git', str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', 'fetch', 'origin', 'main'], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'reset', '--hard', 'origin/main'], cwd=REPO_DIR, check=True)
subprocess.run(['python', '-m', 'pip', 'install', '-e', str(REPO_DIR), '-q'], check=True)

if not RUN4_RAW_CDS.exists():
    subprocess.run([
        'python', str(REPO_DIR / 'scripts' / '1_data_preparation' / 'fetch_sgn_cds.py'),
        '--output', str(RUN4_RAW_CDS),
    ], cwd=REPO_DIR, check=True)

mixed_script = REPO_DIR / 'scripts' / '1_data_preparation' / 'generate_run4b_mixed_pseudolabels.py'
cmd = [
    'python', str(mixed_script),
    '--input', str(RUN4_RAW_CDS),
    '--input-kind', 'cds',
    '--output', str(RUN4B_DATA_DIR / 'run4b_mixed_pseudolabels.jsonl'),
    '--train-output', str(RUN4B_DATA_DIR / 'run4b_mixed_pseudolabels_train.jsonl'),
    '--eval-output', str(RUN4B_DATA_DIR / 'run4b_mixed_pseudolabels_eval.jsonl'),
    '--train-split', '0.9',
    '--seed', '42',
    '--max-length', '512',
    '--limit', '38754',
]
result = subprocess.run(cmd, cwd=REPO_DIR, check=True, text=True, capture_output=True)
print(result.stdout)
stdout = result.stdout.strip()
run4b_generation_summary = json.loads(stdout[stdout.find('{'):stdout.rfind('}') + 1])
print({'run4b_generation_summary': run4b_generation_summary})


In [ ]:
# Run 4B Cell 2: Build AA FASTA and ensure ESM2 embeddings
import json, subprocess
from pathlib import Path

train_file = RUN4B_DATA_DIR / 'run4b_mixed_pseudolabels_train.jsonl'
eval_file = RUN4B_DATA_DIR / 'run4b_mixed_pseudolabels_eval.jsonl'
run4b_fasta = RUN4B_DATA_DIR / 'run4b_mixed_pseudolabels_aa.fasta'
train_total = 0
eval_total = 0
with train_file.open() as train_handle, eval_file.open() as eval_handle, run4b_fasta.open('w') as fasta_handle:
    for line in train_handle:
        row = json.loads(line)
        fasta_handle.write('>{}\n{}\n'.format(row['protein_id'], row['amino_acid_sequence']))
        train_total += 1
    for line in eval_handle:
        row = json.loads(line)
        fasta_handle.write('>{}\n{}\n'.format(row['protein_id'], row['amino_acid_sequence']))
        eval_total += 1

expected_count = train_total + eval_total
existing_count = len(list(RUN4B_EMBEDDINGS.glob('*.pt')))
print({'expected_embeddings': expected_count, 'existing_embeddings': existing_count})
if existing_count < expected_count:
    try:
        import esm  # noqa: F401
    except ModuleNotFoundError:
        subprocess.run(['python', '-m', 'pip', 'install', 'fair-esm', '-q'], check=True)
    subprocess.run([
        'python', str(REPO_DIR / 'scripts' / '1_data_preparation' / 'extract_per_token_esm2.py'),
        '--fasta', str(run4b_fasta),
        '--output', str(RUN4B_EMBEDDINGS),
        '--model', 'esm2_t6_8M_UR50D',
        '--batch-size', '16',
        '--max-length', '512',
    ], cwd=REPO_DIR, check=True)
else:
    print('Reusing existing per-token embeddings.')


In [ ]:
# Run 4B Cell 3: Materialize smoke configs for Kaggle paths
import yaml

RUN4B_CONFIGS = {
    'config_A': REPO_DIR / 'configs' / 'v3_training_config_run4b_gc1.yml',
    'config_B': REPO_DIR / 'configs' / 'v3_training_config_run4b_gc2.yml',
    'config_C': REPO_DIR / 'configs' / 'v3_training_config_run4b_gc2_low2.yml',
    'config_D': REPO_DIR / 'configs' / 'v3_training_config_run4b_gc2_low2_cai010.yml',
}
RUN4B_MATERIALIZED_CONFIGS = {}
for name, source in RUN4B_CONFIGS.items():
    with source.open() as handle:
        cfg = yaml.safe_load(handle)
    cfg['smoke'] = True
    cfg['training']['max_steps'] = 2000
    cfg['training']['checkpoint_every'] = 2000
    cfg['paths']['embeddings_dir'] = str(RUN4B_EMBEDDINGS)
    cfg['paths']['checkpoint_dir'] = str(RUN4B_EXPORT / name / 'checkpoints')
    cfg['paths']['model_output'] = str(RUN4B_EXPORT / name / 'model')
    cfg['paths']['training_data'] = str(train_file)
    cfg['data']['train_file'] = str(train_file)
    cfg['data']['eval_file'] = str(eval_file)
    out = RUN4B_DIR / f'{name}.yml'
    out.parent.mkdir(parents=True, exist_ok=True)
    with out.open('w') as handle:
        yaml.safe_dump(cfg, handle, sort_keys=False)
    RUN4B_MATERIALIZED_CONFIGS[name] = out
    print(name, {'config': str(out), 'gc_weight': cfg['loss']['gc_weight'], 'gc_lambda_low': cfg['loss']['gc_lambda_low'], 'expected_log_cai_weight': cfg['loss']['expected_log_cai_weight'], 'max_steps': cfg['training']['max_steps']})


In [ ]:
# Run 4B Cell 4: Run smoke grid A/B/C/D and save per-config outputs
import shutil, subprocess

RUN4B_EVAL_LIMIT = 200
for name, config_path in RUN4B_MATERIALIZED_CONFIGS.items():
    result_dir = RUN4B_RESULTS / name
    export_dir = RUN4B_EXPORT / name
    result_dir.mkdir(parents=True, exist_ok=True)
    export_dir.mkdir(parents=True, exist_ok=True)
    loss_log = result_dir / 'run4_loss_log.csv'
    print(f'=== Training {name} ===')
    subprocess.run([
        'python', str(REPO_DIR / 'scripts' / '3_training' / 'train_v3_esm2_bart.py'),
        '--config', str(config_path),
        '--loss-log', str(loss_log),
    ], cwd=REPO_DIR, check=True)
    checkpoint = export_dir / 'model' / 'pytorch_model.pt'
    if not checkpoint.exists():
        raise FileNotFoundError(f'Missing smoke checkpoint: {checkpoint}')
    print(f'=== Evaluating {name} ===')
    eval_cmd = [
        'python', str(REPO_DIR / 'scripts' / '4_evaluation' / 'run4_constrained_eval.py'),
        '--config', str(config_path),
        '--checkpoint', str(checkpoint),
        '--eval-file', str(eval_file),
        '--embeddings-dir', str(RUN4B_EMBEDDINGS),
        '--output-dir', str(result_dir),
    ]
    if RUN4B_EVAL_LIMIT is not None:
        eval_cmd.extend(['--limit', str(RUN4B_EVAL_LIMIT)])
    subprocess.run(eval_cmd, cwd=REPO_DIR, check=True)
    print(name, sorted(path.name for path in result_dir.iterdir()))


In [ ]:
# Run 4B Cell 5: Compare smoke grid results
import subprocess

result_dirs = [str(RUN4B_RESULTS / name) for name in ['config_A', 'config_B', 'config_C', 'config_D']]
subprocess.run([
    'python', str(REPO_DIR / 'scripts' / '4_evaluation' / 'run4b_smoke_compare.py'),
    *result_dirs,
    '--output', str(RUN4B_RESULTS / 'run4b_smoke_summary.json'),
], cwd=REPO_DIR, check=True)
print(RUN4B_RESULTS / 'run4b_smoke_summary.json')


## Run 4B Full Training: Config B

Run full 20,000-step Run 4B training after smoke review approval. This uses Config B and writes exportable model artifacts under `/kaggle/working/factorforge_run4b_drive_export/`.

In [ ]:
# Run 4B Full Cell 1: Setup, refresh repo, and generate full mixed pseudo-labels
import json, shutil, subprocess, yaml
from pathlib import Path

REPO_DIR = Path('/kaggle/working/factorforge')
WORK_DIR = Path(globals().get('WORK_DIR', '/kaggle/working/factorforge_data'))
RUN4B_DIR = WORK_DIR / 'run4b'
RUN4B_DATA_DIR = WORK_DIR / 'data' / 'training'
RUN4B_EMBEDDINGS = WORK_DIR / 'data' / 'embeddings' / 'per_token_run2'
RUN4B_RESULTS = REPO_DIR / 'experiments' / 'results' / 'run4b_full'
RUN4B_DRIVE_DIR = Path('/kaggle/working/factorforge_run4b_drive_export')
RUN4_RAW_CDS = WORK_DIR / 'data' / 'raw' / 'sgn_nbenthamiana_cds.fasta'
for directory in [RUN4B_DIR, RUN4B_DATA_DIR, RUN4B_EMBEDDINGS, RUN4B_RESULTS, RUN4B_DRIVE_DIR, RUN4_RAW_CDS.parent]:
    directory.mkdir(parents=True, exist_ok=True)

if not (REPO_DIR / '.git').exists():
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(['git', 'clone', 'https://github.com/eijex/factorforge.git', str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', 'fetch', 'origin', 'main'], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'reset', '--hard', 'origin/main'], cwd=REPO_DIR, check=True)
subprocess.run(['python', '-m', 'pip', 'install', '-e', str(REPO_DIR), '-q'], check=True)

if not RUN4_RAW_CDS.exists():
    subprocess.run([
        'python', str(REPO_DIR / 'scripts' / '1_data_preparation' / 'fetch_sgn_cds.py'),
        '--output', str(RUN4_RAW_CDS),
    ], cwd=REPO_DIR, check=True)

cmd = [
    'python', str(REPO_DIR / 'scripts' / '1_data_preparation' / 'generate_run4b_mixed_pseudolabels.py'),
    '--input', str(RUN4_RAW_CDS),
    '--input-kind', 'cds',
    '--output', str(RUN4B_DATA_DIR / 'run4b_pseudolabels.jsonl'),
    '--train-output', str(RUN4B_DATA_DIR / 'run4b_pseudolabels_train.jsonl'),
    '--eval-output', str(RUN4B_DATA_DIR / 'run4b_pseudolabels_eval.jsonl'),
    '--train-split', '0.9',
    '--seed', '42',
    '--max-length', '512',
    '--limit', '38754',
]
result = subprocess.run(cmd, cwd=REPO_DIR, check=True, text=True, capture_output=True)
print(result.stdout)
stdout = result.stdout.strip()
run4b_generation_summary = json.loads(stdout[stdout.find('{'):stdout.rfind('}') + 1])
print({'run4b_generation_summary': run4b_generation_summary})


In [ ]:
# Run 4B Full Cell 2: Build AA FASTA and ensure ESM2 embeddings
import json, subprocess
from pathlib import Path

REPO_DIR = Path(globals().get('REPO_DIR', '/kaggle/working/factorforge'))
WORK_DIR = Path(globals().get('WORK_DIR', '/kaggle/working/factorforge_data'))
RUN4B_DIR = Path(globals().get('RUN4B_DIR', WORK_DIR / 'run4b'))
RUN4B_DATA_DIR = Path(globals().get('RUN4B_DATA_DIR', WORK_DIR / 'data' / 'training'))
RUN4B_EMBEDDINGS = Path(globals().get('RUN4B_EMBEDDINGS', WORK_DIR / 'data' / 'embeddings' / 'per_token_run2'))
RUN4B_RESULTS = Path(globals().get('RUN4B_RESULTS', REPO_DIR / 'experiments' / 'results' / 'run4b_full'))
RUN4B_DRIVE_DIR = Path(globals().get('RUN4B_DRIVE_DIR', '/kaggle/working/factorforge_run4b_drive_export'))
for directory in [RUN4B_DIR, RUN4B_DATA_DIR, RUN4B_EMBEDDINGS, RUN4B_RESULTS, RUN4B_DRIVE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

train_file = RUN4B_DATA_DIR / 'run4b_pseudolabels_train.jsonl'
eval_file = RUN4B_DATA_DIR / 'run4b_pseudolabels_eval.jsonl'
run4b_fasta = RUN4B_DATA_DIR / 'run4b_pseudolabels_aa.fasta'
if not train_file.exists() or not eval_file.exists():
    raise FileNotFoundError('Missing Run 4B pseudo-label files. Run Run 4B Full Cell 1 first.')
train_total = 0
eval_total = 0
with train_file.open() as train_handle, eval_file.open() as eval_handle, run4b_fasta.open('w') as fasta_handle:
    for line in train_handle:
        row = json.loads(line)
        fasta_handle.write('>{}\n{}\n'.format(row['protein_id'], row['amino_acid_sequence']))
        train_total += 1
    for line in eval_handle:
        row = json.loads(line)
        fasta_handle.write('>{}\n{}\n'.format(row['protein_id'], row['amino_acid_sequence']))
        eval_total += 1

expected_count = train_total + eval_total
existing_count = len(list(RUN4B_EMBEDDINGS.glob('*.pt')))
print({'expected_embeddings': expected_count, 'existing_embeddings': existing_count})
if existing_count < expected_count:
    try:
        import esm  # noqa: F401
    except ModuleNotFoundError:
        subprocess.run(['python', '-m', 'pip', 'install', 'fair-esm', '-q'], check=True)
    subprocess.run([
        'python', str(REPO_DIR / 'scripts' / '1_data_preparation' / 'extract_per_token_esm2.py'),
        '--fasta', str(run4b_fasta),
        '--output', str(RUN4B_EMBEDDINGS),
        '--model', 'esm2_t6_8M_UR50D',
        '--batch-size', '16',
        '--max-length', '512',
    ], cwd=REPO_DIR, check=True)
else:
    print('Reusing existing per-token embeddings.')


In [ ]:
# Run 4B Full Cell 3: Materialize Config B for Kaggle full training
import yaml
from pathlib import Path

REPO_DIR = Path(globals().get('REPO_DIR', '/kaggle/working/factorforge'))
WORK_DIR = Path(globals().get('WORK_DIR', '/kaggle/working/factorforge_data'))
RUN4B_DIR = Path(globals().get('RUN4B_DIR', WORK_DIR / 'run4b'))
RUN4B_DATA_DIR = Path(globals().get('RUN4B_DATA_DIR', WORK_DIR / 'data' / 'training'))
RUN4B_EMBEDDINGS = Path(globals().get('RUN4B_EMBEDDINGS', WORK_DIR / 'data' / 'embeddings' / 'per_token_run2'))
RUN4B_DRIVE_DIR = Path(globals().get('RUN4B_DRIVE_DIR', '/kaggle/working/factorforge_run4b_drive_export'))
for directory in [RUN4B_DIR, RUN4B_EMBEDDINGS, RUN4B_DRIVE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)
train_file = Path(globals().get('train_file', RUN4B_DATA_DIR / 'run4b_pseudolabels_train.jsonl'))
eval_file = Path(globals().get('eval_file', RUN4B_DATA_DIR / 'run4b_pseudolabels_eval.jsonl'))

config_candidates = [
    REPO_DIR / 'configs' / 'v3_training_config_alpha_run2.yml',
    REPO_DIR / 'configs' / 'v3_training_config_run4b_gc2.yml',
]
RUN4B_CONFIG = next((path for path in config_candidates if path.exists()), None)
if RUN4B_CONFIG is None:
    print('Run4B config file not found; using inline Config B fallback.')
    print('Checked:', ', '.join(str(path) for path in config_candidates))
    run4b_cfg = {
        'experiment': '028-run4b-full-gc2-inline',
        'smoke': False,
        'paths': {
            'data_dir': 'data',
            'embeddings_dir': str(RUN4B_EMBEDDINGS),
            'training_data': str(train_file),
            'checkpoint_dir': str(RUN4B_DRIVE_DIR / 'checkpoints'),
            'model_output': str(RUN4B_DRIVE_DIR / 'model'),
        },
        'esm2': {'model_name': 'esm2_t6_8M_UR50D', 'embedding_dim': 320, 'batch_size': 16},
        'model': {
            'encoder': 'esm2_t6_8M_UR50D',
            'encoder_dim': 320,
            'decoder_layers': 6,
            'd_model': 256,
            'decoder_attention_heads': 8,
            'ffn_dim': 256,
            'dropout': 0.15,
            'vocab_size': 69,
            'max_position_embeddings': 1024,
        },
        'bart': {
            'encoder_dim': 320,
            'd_model': 256,
            'decoder_layers': 6,
            'decoder_attention_heads': 8,
            'ffn_dim': 256,
            'max_position_embeddings': 1024,
            'dropout': 0.15,
        },
        'training': {
            'batch_size': 32,
            'learning_rate': 0.00005,
            'warmup_steps': 500,
            'max_steps': 20000,
            'checkpoint_every': 5000,
            'eval_every': 500,
            'early_stopping_patience': 0,
            'label_smoothing': 0.05,
            'weight_decay': 0.01,
            'seed': 42,
        },
        'loss': {
            'ce_weight': 1.0,
            'gc_weight': 2.0,
            'gc_low': 0.40,
            'gc_high': 0.55,
            'gc_lambda_low': 1.0,
            'gc_lambda_high': 1.0,
            'expected_log_cai_weight': 0.05,
            'tai_weight': 0.0,
            'structure_weight': 0.0,
        },
        'data': {
            'train_file': str(train_file),
            'eval_file': str(eval_file),
            'pseudo_label_path': str(train_file),
            'pseudo_label_engine': 'run4b_mixed',
            'pseudo_label_profile': 'mixed',
            'target_train_size': 30000,
            'max_protein_length': 512,
            'train_split': 0.9,
            'val_split': 0.1,
            'test_split': 0.0,
            'require_synonym_mask': True,
            'require_validator_pass': True,
        },
        'safety': {
            'full_training_requires_approval': True,
            'v2_fallback_required': True,
            'constrained_decoding_required': True,
            'wet_lab_claims_allowed': False,
        },
    }
else:
    print('Using Run4B config:', RUN4B_CONFIG)
    with RUN4B_CONFIG.open() as handle:
        run4b_cfg = yaml.safe_load(handle)
run4b_cfg['paths']['embeddings_dir'] = str(RUN4B_EMBEDDINGS)
run4b_cfg['paths']['checkpoint_dir'] = str(RUN4B_DRIVE_DIR / 'checkpoints')
run4b_cfg['paths']['model_output'] = str(RUN4B_DRIVE_DIR / 'model')
run4b_cfg['paths']['training_data'] = str(train_file)
run4b_cfg['data']['train_file'] = str(train_file)
run4b_cfg['data']['eval_file'] = str(eval_file)
run4b_cfg['data']['pseudo_label_path'] = str(train_file)
RUN4B_CONFIG_MATERIALIZED = RUN4B_DIR / 'v3_training_config_run4b_gc2.kaggle.yml'
with RUN4B_CONFIG_MATERIALIZED.open('w') as handle:
    yaml.safe_dump(run4b_cfg, handle, sort_keys=False)
print({
    'config': str(RUN4B_CONFIG_MATERIALIZED),
    'max_steps': run4b_cfg['training']['max_steps'],
    'gc_weight': run4b_cfg['loss']['gc_weight'],
    'gc_lambda_low': run4b_cfg['loss']['gc_lambda_low'],
    'expected_log_cai_weight': run4b_cfg['loss']['expected_log_cai_weight'],
})


In [ ]:
# Run 4B Full Cell 4: Train Config B full run and save loss log
import subprocess
from pathlib import Path

REPO_DIR = Path(globals().get('REPO_DIR', '/kaggle/working/factorforge'))
WORK_DIR = Path(globals().get('WORK_DIR', '/kaggle/working/factorforge_data'))
RUN4B_DIR = Path(globals().get('RUN4B_DIR', WORK_DIR / 'run4b'))
RUN4B_DRIVE_DIR = Path(globals().get('RUN4B_DRIVE_DIR', '/kaggle/working/factorforge_run4b_drive_export'))
RUN4B_CONFIG_MATERIALIZED = Path(globals().get('RUN4B_CONFIG_MATERIALIZED', RUN4B_DIR / 'v3_training_config_run4b_gc2.kaggle.yml'))
if not RUN4B_CONFIG_MATERIALIZED.exists():
    raise FileNotFoundError('Missing materialized Run 4B config. Run Run 4B Full Cell 3 first.')
RUN4B_DRIVE_DIR.mkdir(parents=True, exist_ok=True)

loss_log = RUN4B_DRIVE_DIR / 'run4b_loss_log.csv'
subprocess.run([
    'python', str(REPO_DIR / 'scripts' / '3_training' / 'train_v3_esm2_bart.py'),
    '--config', str(RUN4B_CONFIG_MATERIALIZED),
    '--loss-log', str(loss_log),
], cwd=REPO_DIR, check=True)
print({'model': str(RUN4B_DRIVE_DIR / 'model' / 'pytorch_model.pt'), 'loss_log': str(loss_log)})


In [ ]:
# Run 4B Full Backup Cell: zip checkpoint/logs and upload to HuggingFace Hub
# ⚠️ RUN THIS CELL IMMEDIATELY AFTER CELL 4 FINISHES — BEFORE CLOSING THE SESSION
# If HF_TOKEN is set as a Kaggle Secret, this cell auto-uploads to HuggingFace Hub.
# If not, click "Save Version" (Commit) in the top-right before closing this notebook.
import shutil
from pathlib import Path

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = None

RUN4B_DRIVE_DIR = Path(globals().get("RUN4B_DRIVE_DIR", "/kaggle/working/factorforge_run4b_drive_export"))
if not (RUN4B_DRIVE_DIR / "model" / "pytorch_model.pt").exists():
    raise FileNotFoundError("Missing trained model. Run Run 4B Full Cell 4 first.")

zip_base = Path("/kaggle/working/run4b_drive_export_backup")
zip_path = Path(shutil.make_archive(str(zip_base), "zip", RUN4B_DRIVE_DIR))
print("Backup zip:", zip_path, "size:", zip_path.stat().st_size if zip_path.exists() else None)

if HF_TOKEN:
    try:
        from huggingface_hub import HfApi
        # Repo name format: eijex/factorforge-run4b-checkpoint (private dataset)
        HF_REPO_ID = globals().get("HF_REPO_ID", "eijex/factorforge-run4b-checkpoint")
        api = HfApi(token=HF_TOKEN)
        # Create repo if it doesn't exist (private by default)
        try:
            api.create_repo(HF_REPO_ID, repo_type="dataset", private=True, exist_ok=True)
        except Exception as e:
            print("Repo create warning (may already exist):", e)
        api.upload_file(
            path_or_fileobj=str(zip_path),
            path_in_repo="run4b_drive_export_backup.zip",
            repo_id=HF_REPO_ID,
            repo_type="dataset",
            commit_message="Run 4B checkpoint backup",
        )
        print("✅ Uploaded to HuggingFace Hub:", HF_REPO_ID)
    except Exception as e:
        print("⚠️  HF upload failed:", e)
        print("→ Click 'Save Version' (Commit) NOW before closing this session!")
else:
    print("=" * 60)
    print("⚠️  HF_TOKEN not found in Kaggle Secrets.")
    print("→ Click 'Save Version' (Commit) in the top-right NOW")
    print("→ OR add HF_TOKEN to Kaggle Secrets for automatic upload")
    print("=" * 60)
    print("Backup zip is at:", zip_path)
    print("After Save Version, go to notebook Output tab to download it.")


In [ ]:
# Run 4B Full Cell 5: Full eval set constrained decode and save outputs
import shutil, subprocess
from pathlib import Path

REPO_DIR = Path(globals().get('REPO_DIR', '/kaggle/working/factorforge'))
WORK_DIR = Path(globals().get('WORK_DIR', '/kaggle/working/factorforge_data'))
RUN4B_DIR = Path(globals().get('RUN4B_DIR', WORK_DIR / 'run4b'))
RUN4B_DATA_DIR = Path(globals().get('RUN4B_DATA_DIR', WORK_DIR / 'data' / 'training'))
RUN4B_EMBEDDINGS = Path(globals().get('RUN4B_EMBEDDINGS', WORK_DIR / 'data' / 'embeddings' / 'per_token_run2'))
RUN4B_RESULTS = Path(globals().get('RUN4B_RESULTS', REPO_DIR / 'experiments' / 'results' / 'run4b_full'))
RUN4B_DRIVE_DIR = Path(globals().get('RUN4B_DRIVE_DIR', '/kaggle/working/factorforge_run4b_drive_export'))
RUN4B_CONFIG_MATERIALIZED = Path(globals().get('RUN4B_CONFIG_MATERIALIZED', RUN4B_DIR / 'v3_training_config_run4b_gc2.kaggle.yml'))
eval_file = Path(globals().get('eval_file', RUN4B_DATA_DIR / 'run4b_pseudolabels_eval.jsonl'))
RUN4B_RESULTS.mkdir(parents=True, exist_ok=True)
if not RUN4B_CONFIG_MATERIALIZED.exists():
    raise FileNotFoundError('Missing materialized Run 4B config. Run Run 4B Full Cell 3 first.')
if not eval_file.exists():
    raise FileNotFoundError('Missing Run 4B eval pseudo-label file. Run Run 4B Full Cell 1 first.')

checkpoint = RUN4B_DRIVE_DIR / 'model' / 'pytorch_model.pt'
if not checkpoint.exists():
    raise FileNotFoundError(f'Missing trained model checkpoint: {checkpoint}')
subprocess.run([
    'python', str(REPO_DIR / 'scripts' / '4_evaluation' / 'run4_constrained_eval.py'),
    '--config', str(RUN4B_CONFIG_MATERIALIZED),
    '--checkpoint', str(checkpoint),
    '--eval-file', str(eval_file),
    '--embeddings-dir', str(RUN4B_EMBEDDINGS),
    '--output-dir', str(RUN4B_RESULTS),
], cwd=REPO_DIR, check=True)
shutil.copy2(RUN4B_DRIVE_DIR / 'run4b_loss_log.csv', RUN4B_RESULTS / 'run4b_loss_log.csv')
for name in ['run4b_loss_log.csv', 'run4_comparison.csv', 'run4_summary.json']:
    path = RUN4B_RESULTS / name
    print(path, path.exists(), path.stat().st_size if path.exists() else None)


In [ ]:
# Run 4B Re-decode: save v3 decoded CDS for codon identity analysis
import csv, importlib.util, json, shutil, subprocess, sys, zipfile
from pathlib import Path

import torch

REPO_DIR = Path(globals().get('REPO_DIR', '/kaggle/working/factorforge'))
WORK_DIR = Path(globals().get('WORK_DIR', '/kaggle/working/factorforge_data'))
RUN4B_DIR = Path(globals().get('RUN4B_DIR', WORK_DIR / 'run4b'))
RUN4B_DATA_DIR = Path(globals().get('RUN4B_DATA_DIR', WORK_DIR / 'data' / 'training'))
RUN4B_EMBEDDINGS = Path(globals().get('RUN4B_EMBEDDINGS', WORK_DIR / 'data' / 'embeddings' / 'per_token_run2'))
RUN4B_RESULTS = Path(globals().get('ALPHA_RUN2_RESULTS', REPO_DIR / 'experiments' / 'results' / 'alpha_run2'))
RUN4B_DRIVE_DIR = Path(globals().get('RUN4B_DRIVE_DIR', '/kaggle/working/factorforge_run4b_drive_export'))
RUN4B_CONFIG_MATERIALIZED = Path(globals().get('RUN4B_CONFIG_MATERIALIZED', RUN4B_DIR / 'v3_training_config_run4b_gc2.kaggle.yml'))
eval_file = Path(globals().get('eval_file', RUN4B_DATA_DIR / 'run4b_pseudolabels_eval.jsonl'))
output_csv = RUN4B_RESULTS / 'v3_decoded_eval.csv'

def ensure_factorforge_repo(repo_dir):
    candidates = [
        repo_dir,
        Path.cwd(),
        Path('/kaggle/working/factorforge'),
        Path('/kaggle/working/FactorForge'),
    ]
    for candidate in candidates:
        if (candidate / 'src' / 'factorforge').exists():
            return candidate
    if not (repo_dir / '.git').exists():
        repo_dir.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(['git', 'clone', 'https://github.com/eijex/factorforge.git', str(repo_dir)], check=True)
    if not (repo_dir / 'src' / 'factorforge').exists():
        raise FileNotFoundError(f'FactorForge package not found under {repo_dir}. Run the repo setup cell first or set REPO_DIR correctly.')
    return repo_dir

REPO_DIR = ensure_factorforge_repo(REPO_DIR)
if (REPO_DIR / '.git').exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', 'main'], check=False)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', 'main'], check=False)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', 'main'], check=False)
if not (REPO_DIR / 'src' / 'factorforge' / 'engines').exists():
    raise FileNotFoundError(f'FactorForge engines package missing at {REPO_DIR / "src" / "factorforge" / "engines"}. Check that the cloned repo is eijex/factorforge main.')
repo_src = str(REPO_DIR / 'src')
train_scripts = str(REPO_DIR / 'scripts' / '3_training')
sys.path = [repo_src, train_scripts] + [p for p in sys.path if p not in {repo_src, train_scripts}]
for module_name in list(sys.modules):
    if module_name == 'factorforge' or module_name.startswith('factorforge.'):
        del sys.modules[module_name]
package_dir = REPO_DIR / 'src' / 'factorforge'
package_init = package_dir / '__init__.py'
if not package_init.exists():
    raise FileNotFoundError(f'Missing local factorforge __init__.py: {package_init}')
spec = importlib.util.spec_from_file_location(
    'factorforge',
    package_init,
    submodule_search_locations=[str(package_dir)],
)
factorforge = importlib.util.module_from_spec(spec)
sys.modules['factorforge'] = factorforge
assert spec.loader is not None
spec.loader.exec_module(factorforge)
print('Using factorforge from:', getattr(factorforge, '__file__', None))
print('FactorForge package path:', list(getattr(factorforge, '__path__', [])))
from factorforge.engines.v3.inference.constrained_decoder import constrained_greedy_decode
from factorforge.engines.v3.metrics import load_codon_usage_table
from factorforge.engines.v3.tokenizer import CodonTokenizer
from factorforge.ml.metrics import calculate_cai, calculate_gc
from factorforge.utils.validation import validate_candidate_sequence
training_dir = REPO_DIR / 'scripts' / '3_training'
dataset_module_path = training_dir / 'dataset.py'
if not dataset_module_path.exists():
    raise FileNotFoundError(f'Missing dataset module: {dataset_module_path}')
dataset_spec = importlib.util.spec_from_file_location('dataset', dataset_module_path)
dataset_module = importlib.util.module_from_spec(dataset_spec)
sys.modules['dataset'] = dataset_module
assert dataset_spec.loader is not None
dataset_spec.loader.exec_module(dataset_module)
train_module_path = training_dir / 'train_v3_esm2_bart.py'
if not train_module_path.exists():
    raise FileNotFoundError(f'Missing training module: {train_module_path}')
train_spec = importlib.util.spec_from_file_location('train_v3_esm2_bart', train_module_path)
train_module = importlib.util.module_from_spec(train_spec)
sys.modules['train_v3_esm2_bart'] = train_module
assert train_spec.loader is not None
train_spec.loader.exec_module(train_module)
build_model = train_module.build_model
load_config = train_module.load_config

def find_run4b_checkpoint():
    explicit_candidates = [
        RUN4B_DRIVE_DIR / 'model' / 'pytorch_model.pt',
        RUN4B_DRIVE_DIR / 'checkpoint_best.pt',
        RUN4B_DRIVE_DIR / 'pytorch_model.pt',
        Path('/kaggle/input/factorforge-run4b-drive-export/model/pytorch_model.pt'),
        Path('/kaggle/input/factorforge-run4b-drive-export/checkpoint_best.pt'),
        Path('/kaggle/input/factorforge-run4b-drive-export/pytorch_model.pt'),
        Path('/kaggle/input/factorforge-run4b/checkpoint_best.pt'),
        Path('/kaggle/input/factorforge-run4b/pytorch_model.pt'),
        Path('/kaggle/working/drive/MyDrive/factorforge/run4b/model/pytorch_model.pt'),
        Path('/kaggle/working/drive/MyDrive/factorforge/run4b/checkpoint_best.pt'),
        Path('/kaggle/working/drive/MyDrive/factorforge/run4b/pytorch_model.pt'),
    ]
    for path in explicit_candidates:
        if path.exists():
            return path, explicit_candidates

    # If the Run4B backup was uploaded as a zip dataset, extract it once.
    zip_candidates = []
    for root in [Path('/kaggle/input'), Path('/kaggle/working')]:
        if root.exists():
            zip_candidates.extend(root.rglob('*run4b*export*.zip'))
            zip_candidates.extend(root.rglob('*run4b*backup*.zip'))
    for zip_path in zip_candidates[:10]:
        extract_dir = RUN4B_DRIVE_DIR
        extract_dir.mkdir(parents=True, exist_ok=True)
        print('Extracting Run4B checkpoint archive:', zip_path, '->', extract_dir)
        with zipfile.ZipFile(zip_path) as archive:
            archive.extractall(extract_dir)
        for relative in ['model/pytorch_model.pt', 'checkpoint_best.pt', 'pytorch_model.pt']:
            path = extract_dir / relative
            if path.exists():
                return path, explicit_candidates + zip_candidates

    # Try HuggingFace Hub if HF_TOKEN is available
    try:
        from kaggle_secrets import UserSecretsClient
        hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        hf_token = None
    if hf_token:
        try:
            from huggingface_hub import hf_hub_download
            HF_REPO_ID = globals().get("HF_REPO_ID", "eijex/factorforge-run4b-checkpoint")
            local_zip = RUN4B_DRIVE_DIR / "run4b_hf_download.zip"
            print("Downloading Run4B checkpoint from HuggingFace Hub:", HF_REPO_ID)
            hf_hub_download(
                repo_id=HF_REPO_ID,
                filename="run4b_drive_export_backup.zip",
                repo_type="dataset",
                token=hf_token,
                local_dir=str(local_zip.parent),
                local_dir_use_symlinks=False,
            )
            downloaded_zip = RUN4B_DRIVE_DIR / "run4b_drive_export_backup.zip"
            if downloaded_zip.exists():
                print("Extracting HF checkpoint archive:", downloaded_zip, "->", RUN4B_DRIVE_DIR)
                import zipfile as _zipfile
                with _zipfile.ZipFile(downloaded_zip) as archive:
                    archive.extractall(RUN4B_DRIVE_DIR)
                for relative in ["model/pytorch_model.pt", "checkpoint_best.pt", "pytorch_model.pt"]:
                    path = RUN4B_DRIVE_DIR / relative
                    if path.exists():
                        return path, []
        except Exception as e:
            print("HF Hub download failed:", e)

    # Last resort: scan common Kaggle roots for likely checkpoint file names.
    scanned = list(explicit_candidates) + zip_candidates
    for root in [Path('/kaggle/input'), Path('/kaggle/working')]:
        if not root.exists():
            continue
        for name in ['pytorch_model.pt', 'checkpoint_best.pt']:
            matches = sorted(root.rglob(name), key=lambda p: p.stat().st_size if p.exists() else 0, reverse=True)
            scanned.extend(matches[:20])
            if matches:
                return matches[0], scanned
    return None, scanned

checkpoint, checked_paths = find_run4b_checkpoint()
if checkpoint is None:
    preview = '\n'.join(str(path) for path in checked_paths[:80])
    raise FileNotFoundError('Run4B checkpoint not found. Attach/upload the Run4B export dataset or backup zip. Checked paths:\n' + preview)
print('Using Run4B checkpoint:', checkpoint)
if not RUN4B_CONFIG_MATERIALIZED.exists():
    raise FileNotFoundError(f'Missing Run4B materialized config: {RUN4B_CONFIG_MATERIALIZED}')
if not eval_file.exists():
    raise FileNotFoundError(f'Missing eval JSONL: {eval_file}')
if not RUN4B_EMBEDDINGS.exists():
    raise FileNotFoundError(f'Missing eval embeddings directory: {RUN4B_EMBEDDINGS}. Re-run embedding extraction first.')

def read_jsonl(path):
    with path.open(encoding='utf-8') as handle:
        return [json.loads(line) for line in handle if line.strip()]

def protein_sequence(row):
    seq = row.get('amino_acid_sequence') or row.get('protein_sequence') or row.get('sequence')
    if not seq:
        raise ValueError(f"Missing protein sequence for {row.get('protein_id')}")
    return ''.join(seq.upper().split()).rstrip('*')

cfg = load_config(str(RUN4B_CONFIG_MATERIALIZED))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = CodonTokenizer.default()
table = load_codon_usage_table()
model = build_model(cfg).to(device)
state = torch.load(checkpoint, map_location=device, weights_only=True)
if isinstance(state, dict) and 'model_state_dict' in state:
    state = state['model_state_dict']
model.load_state_dict(state)
model.eval()

rows = read_jsonl(eval_file)
RUN4B_RESULTS.mkdir(parents=True, exist_ok=True)
decoded_rows = []
gc_values = []
cai_values = []
validator_passed = 0
for index, row in enumerate(rows, start=1):
    protein_id = str(row['protein_id'])
    protein = protein_sequence(row)
    embedding_file = RUN4B_EMBEDDINGS / f'{protein_id}.pt'
    if not embedding_file.exists():
        raise FileNotFoundError(f'Missing embedding for {protein_id}: {embedding_file}')
    embedding = torch.load(embedding_file, map_location=device, weights_only=True)['embeddings']
    with torch.no_grad():
        token_ids = constrained_greedy_decode(
            model,
            embedding.to(device).unsqueeze(0),
            protein,
            tokenizer,
        )
    v3_cds = tokenizer.decode(token_ids.squeeze(0).tolist(), skip_special_tokens=True)
    validator = validate_candidate_sequence(protein, v3_cds)
    gc = calculate_gc(v3_cds)
    cai = calculate_cai(v3_cds, table.codon_weights)
    gc_values.append(gc)
    cai_values.append(cai)
    validator_passed += 1 if validator['passed'] else 0
    decoded_rows.append({
        'protein_id': protein_id,
        'amino_acid_sequence': protein,
        'v3_decoded_cds': v3_cds,
        'validator_passed': validator['passed'],
    })
    if index == 1 or index % 250 == 0 or index == len(rows):
        print(f'decoded {index}/{len(rows)}')

with output_csv.open('w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=['protein_id', 'amino_acid_sequence', 'v3_decoded_cds', 'validator_passed'])
    writer.writeheader()
    writer.writerows(decoded_rows)

summary = {
    'decoded_count': len(decoded_rows),
    'validator_pass_rate': validator_passed / len(decoded_rows) if decoded_rows else 0.0,
    'mean_gc': sum(gc_values) / len(gc_values) if gc_values else 0.0,
    'mean_cai': sum(cai_values) / len(cai_values) if cai_values else 0.0,
    'output_csv': str(output_csv),
    'checkpoint': str(checkpoint),
}
print(json.dumps(summary, indent=2, sort_keys=True))
